## Imports

In [2]:
import json
import pandas as pd
import random
import string

## Import list of posts
I get all of the posts that have been identified as a breach. I'm going to get the id's of them so I can reduce the load of the zst reading.

In [8]:
breach_posts: pd.DataFrame = pd.read_csv("final_breach_posts.csv")
post_ids: list[str] = []

for post in breach_posts.itertuples():
    post_ids.append(post[5])

post_id_frame = pd.DataFrame(post_ids, columns=['id'])
post_id_frame.to_csv("breach_post_ids.csv")

## Consolidate jsonl files
Because each month generates its own jsonl file, combine all of the relevant data into one big file.

In [4]:
import glob
from pathlib import Path
import os

input_directory = "/scratch/jacobli/comments/"
output_file = "all_comments.jsonl"

jsonl_files = glob.glob(os.path.join(input_directory, "*.jsonl"))
jsonl_files.sort()

print(f"Found {len(jsonl_files)} JSONL files")

with open(output_file, 'w', encoding='utf-8') as outfile:
    for jsonl_file in jsonl_files:
        print(f"Processing: {jsonl_file}")
        with open(jsonl_file, 'r', encoding='utf-8') as infile:
            for line in infile:
                outfile.write(line)

Found 12 JSONL files
Processing: /scratch/jacobli/comments/output1.jsonl
Processing: /scratch/jacobli/comments/output10.jsonl
Processing: /scratch/jacobli/comments/output11.jsonl
Processing: /scratch/jacobli/comments/output12.jsonl
Processing: /scratch/jacobli/comments/output2.jsonl
Processing: /scratch/jacobli/comments/output3.jsonl
Processing: /scratch/jacobli/comments/output4.jsonl
Processing: /scratch/jacobli/comments/output5.jsonl
Processing: /scratch/jacobli/comments/output6.jsonl
Processing: /scratch/jacobli/comments/output7.jsonl
Processing: /scratch/jacobli/comments/output8.jsonl
Processing: /scratch/jacobli/comments/output9.jsonl


## Process data
The zst files will already be filtered by id, so all I have to do is convert it into a readable format with the right data. I do not need to apply any filtering.

In [5]:
comment_data = []
comments_input: str = "all_comments.jsonl"
comment_records = []

with open(comments_input, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            comment_data.append(json.loads(line))

for comment in comment_data:
    content: str = comment["body"]
    if not content or content in ["[removed]", "[deleted]"]:
        continue
    comment_records.append({
        "author": comment["author"],
        "time_of_comment": comment["created"],
        "subreddit": comment["subreddit"],
        "comment_post": comment["link_id"][3:],
        "comment_link": comment["id"],
        "content": content,
        "num_upvotes": comment["ups"],
        "num_downvotes": comment["downs"],
        "parent_comment": comment["parent_id"][3:],
        "comment_length": len(content),
        "controversial": (comment["controversiality"] != 0)
    })
        

comments: pd.DataFrame = pd.DataFrame(comment_records)
comments.to_csv("breach_post_comments.csv", index=False)

## Format the data
Now that I have the ids and content, make it look nice for review. This section of code will output a visual diagram that is easily understandable.

In [6]:
import csv
import re
import os
from collections import defaultdict

def load_posts(posts_folder):
    posts = {}
    for filename in os.listdir(posts_folder):
        if filename.endswith('.csv'):
            filepath = os.path.join(posts_folder, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                reader = csv.reader(f)
                next(reader)
                for row in reader:
                    if len(row) >= 8:
                        post_id = row[4].strip()
                        title = row[6].strip()
                        content = row[7].strip()
                        
                        content = re.sub(r'\n+', ' ', content)
                        
                        posts[post_id] = {
                            'title': title,
                            'content': content
                        }
    
    return posts

def build_tree(csv_file):
    tree = defaultdict(list)
    comment_content = {}
    
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            comment_post = row['comment_post'].strip()
            comment_link = row['comment_link'].strip()
            parent_comment = row['parent_comment'].strip()
            content = row['content'].strip()
            
            comment_content[comment_link] = content
            if not parent_comment or parent_comment == comment_post:
                if comment_link not in tree[comment_post]:
                    tree[comment_post].append(comment_link)
            else:
                if comment_link not in tree[parent_comment]:
                    tree[parent_comment].append(comment_link)
    
    return tree, comment_content

def print_tree(tree, comment_content, node, file, prefix="", is_last=True):
    connector = "└─────> " if is_last else "├─────> "
    content = comment_content.get(node, "")
    content = re.sub(r'\n+', ' ', content)
    print(f"{prefix}{connector}{node}: {content}", file=file)
    children = tree.get(node, [])
    extension = "        " if is_last else "│       "
    new_prefix = prefix + extension
    for i, child in enumerate(children):
        is_last_child = (i == len(children) - 1)
        print_tree(tree, comment_content, child, file, new_prefix, is_last_child)

def display_all_trees(comments_file, posts_folder, output_file="comment_tree.txt"):
    posts = load_posts(posts_folder)
    tree, comment_content = build_tree(comments_file)
    all_nodes = set(tree.keys())
    all_children = set()
    for children in tree.values():
        all_children.update(children)
    roots = all_nodes - all_children
    with open(output_file, 'w', encoding='utf-8') as f:
        for root in sorted(roots):
            post_info = posts.get(root, {'title': 'Unknown', 'content': ''})
            print(post_info['title'], file=f)
            print(f"{root}: {post_info['content']}", file=f)
            print("|", file=f)
            children = tree.get(root, [])
            for i, child in enumerate(children):
                is_last = (i == len(children) - 1)
                print_tree(tree, comment_content, child, f, "", is_last)
            
            print(file=f)
    
    print(f"Tree written to {output_file}")

In [7]:
csv_file = "breach_post_comments.csv"
display_all_trees(csv_file, "filtered_posts")

Tree written to comment_tree.txt


## Other formatting
This one is more complete and outputs all information about the post to a csv file

In [3]:
import csv
import re
import os
from collections import defaultdict

def load_posts(posts_folder):
    posts = {}
    
    for filename in os.listdir(posts_folder):
        if filename.endswith('.csv'):
            filepath = os.path.join(posts_folder, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                reader = csv.reader(f)
                next(reader)
                for row in reader:
                    if len(row) >= 8:
                        post_id = row[4].strip()
                        title = row[6].strip()
                        content = row[7].strip()
                        
                        # Replace newlines with spaces
                        content = re.sub(r'\n+', ' ', content)
                        
                        posts[post_id] = {
                            'title': title,
                            'content': content
                        }
    
    return posts

def build_tree(csv_file):
    tree = defaultdict(list)
    comment_content = {}
    
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        
        for row in reader:
            comment_post = row['comment_post'].strip()
            comment_link = row['comment_link'].strip()
            parent_comment = row['parent_comment'].strip()
            content = row['content'].strip()
            comment_content[comment_link] = content
            if not parent_comment or parent_comment == comment_post:
                if comment_link not in tree[comment_post]:
                    tree[comment_post].append(comment_link)
            else:
                if comment_link not in tree[parent_comment]:
                    tree[parent_comment].append(comment_link)
    
    return tree, comment_content

def write_tree_to_rows(tree, comment_content, node, rows, depth=0):
    content = comment_content.get(node, "")
    content = re.sub(r'\n+', ' ', content)
    row = [''] * depth + [node, content]
    rows.append(row)
    children = tree.get(node, [])
    for child in children:
        write_tree_to_rows(tree, comment_content, child, rows, depth + 1)

def display_all_trees(comments_file, posts_folder, output_file="comment_tree.csv"):
    posts = load_posts(posts_folder)
    tree, comment_content = build_tree(comments_file)
    all_nodes = set(tree.keys())
    all_children = set()
    for children in tree.values():
        all_children.update(children)
    roots = all_nodes - all_children
    with open(output_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        for root in sorted(roots):
            post_info = posts.get(root, {'title': 'Unknown', 'content': ''})
            writer.writerow([post_info['title']])
            writer.writerow([root, post_info['content']])
            rows = []
            children = tree.get(root, [])
            for child in children:
                write_tree_to_rows(tree, comment_content, child, rows, depth=1)
            for row in rows:
                writer.writerow(row)
            writer.writerow([])
    
    print(f"Tree written to {output_file}")

In [4]:
display_all_trees("breach_post_comments.csv", "filtered_posts", "comment_tree.csv")

Tree written to comment_tree.csv
